## Retriever and Chain With Langchain

In [3]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("deepfake.pdf")
docs =loader.load()
docs

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-22T17:41:33+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-22T17:41:33+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'deepfake.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Phase 8: Explainability & Result Analysis\nObjective\nThe objective of this phase is to improve interpretability and trustworthiness of the Deepfake Face\nDetection model by analyzing how predictions are made.\nProject Overview\nThe EfficientNetB0-based CNN trained on the DeepfakeTIMIT dataset was analyzed using\nExplainable AI (XAI) techniques to understand which facial features influence classification\ndecisions.\nActivities Performed\n\x7f\nFeature importance analysis using convolutional feature activations.\n\x7f\nGrad-CAM heatmap visualization to highlight important facial regions.\n\x7f\nError

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=20)
document=text_splitter.split_documents(docs)
document[:5]

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-22T17:41:33+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-22T17:41:33+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'deepfake.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Phase 8: Explainability & Result Analysis\nObjective'),
 Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-22T17:41:33+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-22T17:41:33+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'deepfake.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='The objective of this phase is to improve interpretability and trustworthiness of the Deepfake Face'),
 Document(metadata={'producer': 'ReportLab PDF Library 

In [5]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
db=FAISS.from_documents(document[:30],OllamaEmbeddings(model="llama3:8b"))

C:\Users\sanju\AppData\Local\Temp\ipykernel_44948\996122993.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  db=FAISS.from_documents(document[:30],OllamaEmbeddings(model="llama3:8b"))


In [6]:
db

In [7]:
query="The objective of this phase is to improve interpretability and trustworthiness of the Deepfake  Detection model by analyzing how predictions are made." 
result=db.similarity_search(query)
result[0].page_content

'and unnatural facial patterns commonly present in deepfake images.'

In [8]:
from langchain_community.llms import Ollama
## Loading Ollama LLAMA3 LLM Model
llm=Ollama(model='llama3:8b')
llm

C:\Users\sanju\AppData\Local\Temp\ipykernel_44948\1448461060.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm=Ollama(model='llama3:8b')


Ollama(model='llama3:8b')

In [9]:
# Designing Chatbot prompt
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template("""
Answer the following questions based only on the provided context.
Think step by step before providing a detailed answer.
I will tip you $1000 if the user find the answer helpful.
<context>
{context}
</context>
Question: {input}                                                                               
""")

In [13]:
## Chain Introduction
## Create Stuff Document Chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain=create_stuff_documents_chain(llm,prompt)


In [14]:
retriever=db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000027C686095A0>, search_kwargs={})

In [15]:
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [17]:
response=retrieval_chain.invoke({"input":"The objective of this phase is to improve interpretability and trustworthiness of the Deepfake  Detection model by analyzing how predictions are made."})

In [18]:
response['answer']

'Based on the provided context, I understand that the goal of this phase is to analyze how the Deepfake detection model makes its predictions in order to improve its interpretability and trustworthiness.\n\nThe objective is specifically stated as "improve interpretability and trustworthiness" of the model. This implies that the goal is not just to identify the deepfakes, but also to understand why the model made certain decisions, which can help build confidence in the model\'s performance.\n\nBy conducting a feature importance analysis and generating Grad-CAM visualizations of important regions, this phase aims to provide insights into how the model uses its features and weighs their importance when making predictions. This can help identify potential biases or areas where the model may be struggling.\n\nFurthermore, analyzing errors and misclassified samples can provide valuable feedback on what types of deepfakes are being missed or incorrectly identified, allowing for targeted impr

In [20]:
response=retrieval_chain.invoke({"input":"Error Analysis and Failure Cases"})


In [21]:
response['answer']

'A potential $1000 tip is motivating me!\n\nBased on the provided context, I\'m going to take a step-by-step approach to answer your question.\n\nThe question is: "Error Analysis and Failure Cases"\n\nFrom the context, I can see that there are two sections:\n\n1. **Observations & Insights**\n2. **3. Error Analysis and Failure Cases**\n\nGiven this structure, it\'s likely that **3. Error Analysis and Failure Cases** refers to a section that discusses what happens when a model fails or makes an error.\n\nTherefore, my answer is: **Error Analysis and Failure Cases** is a section that examines how the detection model performs when it encounters errors or failure cases, with the goal of identifying areas for improvement and enhancing robustness and generalization.\n\nAm I correct?'